# Section 4 — Memorizing Transformers: The KV Twist

Memorizing Transformers don't invent a new memory format. They reuse the exact same mechanism the transformer already uses for attention — but point it at a bank of past context.

In [ ]:
torch.manual_seed(5)

D_MODEL: int = 16
N_LOCAL: int = 6      # tokens in the local window
N_EXT: int = 10       # past (K, V) pairs in external memory bank

# --- local context: Q, K, V for current window ---
local_Q: torch.Tensor = torch.randn(N_LOCAL, D_MODEL)
local_K: torch.Tensor = torch.randn(N_LOCAL, D_MODEL)
local_V: torch.Tensor = torch.randn(N_LOCAL, D_MODEL)

# --- external memory bank: 10 past key-value pairs ---
ext_K: torch.Tensor = torch.randn(N_EXT, D_MODEL)
ext_V: torch.Tensor = torch.randn(N_EXT, D_MODEL)

# --- plant a strong match: local query 3 should retrieve external key 7 ---
ext_K[7] = F.normalize(local_Q[3] + 0.05 * torch.randn(D_MODEL), dim=0) * 3.0

# --- concatenate local and external keys/values ---
K_full: torch.Tensor = torch.cat([local_K, ext_K], dim=0)   # (N_LOCAL + N_EXT, D_MODEL)
V_full: torch.Tensor = torch.cat([local_V, ext_V], dim=0)

# --- compute attention scores for all queries against the full key set ---
attn_scores: torch.Tensor = local_Q @ K_full.T / (D_MODEL ** 0.5)
attn_weights: torch.Tensor = F.softmax(attn_scores, dim=-1)  # (N_LOCAL, N_LOCAL + N_EXT)

# --- visualize: rows = local queries, columns = local + external keys ---
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(
    attn_weights.detach().numpy(), cmap="Blues",
    aspect="auto", vmin=0, vmax=attn_weights.max().item()
)

# column labels
col_labels: list[str] = (
    [f"L{i}" for i in range(N_LOCAL)] +
    [f"E{i}" for i in range(N_EXT)]
)
ax.set_xticks(range(N_LOCAL + N_EXT))
ax.set_xticklabels(col_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(N_LOCAL))
ax.set_yticklabels([f"query_{i}" for i in range(N_LOCAL)])

# vertical dashed separator between local and external
ax.axvline(x=N_LOCAL - 0.5, color="red", linewidth=2, linestyle="--", label="local | external")
ax.text(N_LOCAL - 0.5 + 0.1, -0.7, "← local       external →",
        color="red", fontsize=8, va="top")

ax.set_title(
    "Attention weights: local queries × (local + external) keys\n"
    "query_3 attends strongly to external key E7",
    fontsize=11
)
plt.colorbar(im, ax=ax, label="attention weight")
plt.tight_layout()
plt.show()

# confirm query 3's strongest key
top_col: int = int(attn_weights[3].argmax().item())
print(f"query_3 top attention → column {top_col} = '{col_labels[top_col]}' "
      f"(weight={attn_weights[3, top_col]:.3f})")

In [ ]:
torch.manual_seed(5)

D_MODEL: int = 16
N_LOCAL: int = 6      # tokens in the local window
N_EXT: int = 10       # past (K, V) pairs in external memory bank

# --- local context: Q, K, V for current window ---
local_Q: torch.Tensor = torch.randn(N_LOCAL, D_MODEL)
local_K: torch.Tensor = torch.randn(N_LOCAL, D_MODEL)
local_V: torch.Tensor = torch.randn(N_LOCAL, D_MODEL)

# --- external memory bank: 10 past key-value pairs ---
ext_K: torch.Tensor = torch.randn(N_EXT, D_MODEL)
ext_V: torch.Tensor = torch.randn(N_EXT, D_MODEL)

# --- plant a strong match: local query 3 should retrieve external key 7 ---
ext_K[7] = F.normalize(local_Q[3] + 0.05 * torch.randn(D_MODEL), dim=0) * 3.0

# --- concatenate local and external keys/values ---
K_full: torch.Tensor = torch.cat([local_K, ext_K], dim=0)   # (N_LOCAL + N_EXT, D_MODEL)
V_full: torch.Tensor = torch.cat([local_V, ext_V], dim=0)

# --- compute attention scores for all queries against the full key set ---
attn_scores: torch.Tensor = local_Q @ K_full.T / (D_MODEL ** 0.5)
attn_weights: torch.Tensor = F.softmax(attn_scores, dim=-1)  # (N_LOCAL, N_LOCAL + N_EXT)

# --- visualize: rows = local queries, columns = local + external keys ---
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(
    attn_weights.detach().numpy(), cmap="Blues",
    aspect="auto", vmin=0, vmax=attn_weights.max().item()
)

# column labels
col_labels: list[str] = (
    [f"L{i}" for i in range(N_LOCAL)] +
    [f"E{i}" for i in range(N_EXT)]
)
ax.set_xticks(range(N_LOCAL + N_EXT))
ax.set_xticklabels(col_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(N_LOCAL))
ax.set_yticklabels([f"query_{i}" for i in range(N_LOCAL)])

# vertical dashed separator between local and external
ax.axvline(x=N_LOCAL - 0.5, color="red", linewidth=2, linestyle="--", label="local | external")
ax.text(N_LOCAL - 0.5 + 0.1, -0.7, "← local       external →",
        color="red", fontsize=8, va="top")

ax.set_title(
    "Attention weights: local queries × (local + external) keys\n"
    "query_3 attends strongly to external key E7",
    fontsize=11
)
plt.colorbar(im, ax=ax, label="attention weight")
plt.tight_layout()
plt.show()

# confirm query 3's strongest key
top_col: int = int(attn_weights[3].argmax().item())
print(f"query_3 top attention → column {top_col} = '{col_labels[top_col]}' "
      f"(weight={attn_weights[3, top_col]:.3f})")

# Section 5 — Beyond: Titans, Mamba, and the Spectrum

## 5a — Titans: Learning While Thinking

Titans (Behrouz et al., 2024, Google Research) takes this one step further. Instead of *addressing* a fixed external memory, the long-term memory module in Titans is itself a small neural network — and its weights are updated via gradient descent *during the forward pass*.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(12)

SEQ_LEN: int = 20

# --- synthetic token sequence with a few high-surprise spikes ---
token_values: torch.Tensor = torch.randn(SEQ_LEN) * 0.3
# inject unexpected spikes at positions 5, 13, 17
spike_positions: list[int] = [5, 13, 17]
for pos in spike_positions:
    token_values[pos] = 3.5 + torch.rand(1).item()

# --- compute surprise as prediction error vs. running mean ---
running_mean: float = 0.0
surprise: list[float] = []
alpha: float = 0.3   # exponential moving average coefficient

for t in range(SEQ_LEN):
    val: float = token_values[t].item()
    err: float = (val - running_mean) ** 2    # squared prediction error
    surprise.append(err)
    running_mean = alpha * val + (1 - alpha) * running_mean

surprise_arr: np.ndarray = np.array(surprise)

# --- plot surprise signal, highlight spike positions in red ---
colors: list[str] = [
    "#e53935" if i in spike_positions else "#90caf9"
    for i in range(SEQ_LEN)
]

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(range(SEQ_LEN), surprise_arr, color=colors, edgecolor="#333", width=0.7)
ax.set_xlabel("Token index")
ax.set_ylabel("Surprise (prediction error²)")
ax.set_title(
    "The model's highlighter — surprising tokens get written into long-term memory more strongly",
    fontsize=11
)
ax.set_xticks(range(SEQ_LEN))

# legend patches
import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color="#e53935", label="high-surprise token (spike)"),
    mpatches.Patch(color="#90caf9", label="routine token")
], loc="upper right")

plt.tight_layout()
plt.show()

<details>
<summary>learning during inference — what it means</summary>

In every transformer you have ever used, inference is inference and training is training. The weights are frozen once training is done. The model's 'knowledge' is fixed; the forward pass is just matrix multiplication.

Titans breaks this boundary. The long-term memory module is a small MLP whose weights are updated via gradient descent *during the forward pass*, using the surprise signal as a learning rate modulator. A high-surprise token causes a larger weight update; a routine token causes a smaller one. The memory module is literally learning from the sequence it is currently processing.

This has a radical implication: the model is no longer stateless at inference time. Processing a document changes the memory module's weights. Those weights are the model's accumulated experience of this particular document. The boundary between 'what I was trained on' and 'what I am thinking about right now' becomes blurry — which is arguably closer to how biological cognition works than any previous architecture.

</details>

## 5b — Mamba / SSMs: Compression Without Addressability

Mamba (Gu & Dao, 2023) takes a fundamentally different approach. Instead of an external addressable memory, all history is compressed into a fixed-size hidden state vector that evolves recurrently.

The tradeoff is stark:
- **NTM / Memorizing Transformers**: you can look up a specific past event. Cost: O(N) or O(N²) in memory/compute.
- **Mamba**: linear time, constant memory. But you cannot look up — you can only compress.

Think of NTM as a filing cabinet you can search. Mamba is a single notebook page that gets continuously rewritten — efficient, but lossy.

## 5c — The Spectrum

Placing all mechanisms on two axes: expressiveness (can it store and retrieve specific past events?) vs. computational cost.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# --- mechanism data: (name, description, x=compute_cost, y=addressability, color) ---
mechanisms: list[tuple[str, str, float, float, str]] = [
    ("Mamba / SSM",
     "linear time,\nconstant memory,\nbut lossy",
     1.0, 1.0, "#81c784"),
    ("Memorizing\nTransformer",
     "same attention,\nextended to past KV",
     5.5, 7.5, "#64b5f6"),
    ("Titans",
     "memory = small net,\nupdated at inference",
     5.0, 8.0, "#ff8a65"),
    ("NTM",
     "differentiable RAM,\nfull addressability",
     7.5, 8.5, "#ba68c8"),
    ("DNC",
     "NTM + temporal links\n+ allocation",
     8.5, 9.2, "#f06292"),
    ("Attention /\nKV Cache",
     "standard transformer",
     4.0, 5.5, "#ffd54f"),
]

fig, ax = plt.subplots(figsize=(11, 7))

# --- shaded region for 'where most current LLMs live' ---
llm_patch = mpatches.FancyBboxPatch(
    (2.5, 4.0), 3.5, 3.0,
    boxstyle="round,pad=0.3",
    facecolor="#fff9c4", edgecolor="#f9a825",
    linewidth=2, linestyle="--", zorder=1
)
ax.add_patch(llm_patch)
ax.text(4.25, 7.15, "where most current LLMs live",
        ha="center", va="bottom", fontsize=8, color="#f57f17", style="italic")

# --- plot each mechanism ---
for (name, desc, x, y, color) in mechanisms:
    ax.scatter(x, y, s=320, color=color, edgecolors="#333",
               linewidths=1.5, zorder=5)
    ax.annotate(
        f"{name}\n{desc}",
        xy=(x, y), xytext=(x + 0.25, y - 0.7),
        fontsize=7.5, color="#222",
        arrowprops=dict(arrowstyle="-", color="#999", lw=0.8),
        zorder=6
    )

ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_xlabel("Computational cost at inference  →", fontsize=11)
ax.set_ylabel("Memory expressiveness / addressability  →", fontsize=11)
ax.set_title(
    "The latent memory spectrum: expressiveness vs. computational cost",
    fontsize=13, pad=12
)
ax.set_xticks([])
ax.set_yticks([])
ax.text(0.15, 0.5, "Low", fontsize=9, color="#777", va="center")
ax.text(9.7, 0.5, "High", fontsize=9, color="#777", va="center", ha="right")
ax.text(0.15, 0.5, "Low", fontsize=9, color="#777", va="center")
ax.text(0.15, 9.5, "High", fontsize=9, color="#777", va="top")

ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

<details>
<summary>where research is going</summary>

The spectrum is not a ranking where higher-right is always better. Different points are optimal for different tasks. Long-document question answering needs addressable memory; real-time streaming classification needs constant-memory efficiency.

The current frontier is *hybrid architectures* that combine multiple mechanisms in the same model. Jamba (AI21 Labs, 2024) interleaves Mamba SSM layers with standard attention layers, capturing the efficiency of recurrent compression for long-range dependencies while preserving the precision of attention for local context. The intuition: let SSM handle the 'gist' of distant history, and attention handle the details of the current window.

Titans takes a different hybrid approach: a fast attention window + a slow memory module that learns from the sequence. The 'fast' and 'slow' framing echoes the dual-process theories of cognition (Kahneman's System 1 / System 2) — though the analogy should not be pushed too far.

The unifying question for the next few years: can we get addressable retrieval at linear cost? Current evidence suggests no, but the gap is narrowing. Every point on the spectrum is a live research program.

</details>